# APIM ❤️ Responsible AI

## Content Safety lab — sobre el APIM COMPARTIDO
![flow](../../images/content-safety.gif)

Playground para probar la [content safety policy](https://learn.microsoft.com/en-us/azure/api-management/llm-content-safety-policy). La policy aplica controles de content safety sobre cualquier prompt de LLM, enviándolo al servicio [Azure AI Content Safety](https://learn.microsoft.com/en-us/azure/ai-services/content-safety/overview) antes de llegar al backend del modelo. Cuando la policy detecta contenido problemático, API Management bloquea la solicitud, devuelve un `403` y explica **qué categoría y qué nivel de severidad la dispararon** (ver `policy.xml`).

⚠️ **Este lab ya NO crea su propia instancia de APIM.** Usa la APIM COMPARTIDA del lab [`shared-apim-gateway`](../shared-apim-gateway/README.md) — corre ese lab primero (una sola vez) y toma sus outputs (`resourceGroupName`, `apimServiceName`) para la celda `0️⃣` de aquí abajo.

[Ver configuración de la policy](policy.xml)

### Prerequisites

- [Python 3.12 o superior](https://www.python.org/) instalado
- [VS Code](https://code.visualstudio.com/) con la [extensión de Jupyter](https://marketplace.visualstudio.com/items?itemName=ms-toolsai.jupyter)
- [uv](https://docs.astral.sh/uv/) — ejecuta `uv sync` desde la raíz del repo para instalar dependencias
- [Una suscripción de Azure](https://azure.microsoft.com/free/) con roles [Contributor](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#contributor) + [RBAC Administrator](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#role-based-access-control-administrator) u [Owner](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#owner)
- [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli) instalado y con sesión iniciada (`az login`)
- **El lab `shared-apim-gateway` ya desplegado** — necesitas su `resourceGroupName` y `apimServiceName`

▶️ Ejecuta `Run All`, o paso a paso...


<a id='0'></a>
### 0️⃣ Inicializar variables del notebook

- Los recursos propios de este lab (Content Safety, Foundry) se sufijan con un string único basado en tu subscription id.
- **`shared_apim_resource_group_name` / `shared_apim_service_name`**: pega aquí los outputs del lab `shared-apim-gateway` (`resourceGroupName` y `apimServiceName`). Este lab YA NO crea su propia APIM — se conecta a esa instancia compartida.
- **`shared_apim_logger_id`**: opcional — output `apimLoggerId` del lab compartido. Conecta la API de este lab al mismo Application Insights que usan los demás labs.
- **`lab_prefix`**: el nombre propio de este lab dentro del APIM compartido — por defecto toma el nombre de la carpeta del lab (`deployment_name`, es decir `content-safety`). Con esto la API que crea este lab queda **completamente independiente**: su propio path (`/content-safety/...`), su propio backend, product y subscriptions — no comparte ni interfiere con las APIs de otros labs sobre la misma instancia.
- **`shared_log_analytics_workspace_id`**: opcional — resource ID del Log Analytics Workspace del lab compartido, para que los logs de Content Safety de este lab caigan en el mismo workspace que usan todos los demás labs. Déjalo vacío si aún no lo expones como output ahí.
- Ajusta modelo/versión según la [disponibilidad por región](https://learn.microsoft.com/azure/ai-services/openai/concepts/models). Este lab usa **2 modelos** (`gpt-5.6-luna` y `Phi-4`); las pruebas de más abajo llaman a los **2**, no solo al primero.


In [1]:
import os, sys, json
sys.path.insert(1, '../../shared')  # add the shared directory to the Python path
import utils

deployment_name = os.path.basename(os.path.dirname(globals()['__vsc_ipynb_file__']))
resource_group_name = f"lab-{deployment_name}-V3"  # change the name to match your naming style
resource_group_location = "swedencentral"

# --- APIM COMPARTIDO (outputs del lab shared-apim-gateway) --------------------
# ⚠️ Reemplaza estos 2 valores con los outputs reales de ese lab.
shared_apim_resource_group_name = "rg-shared-apim-gateway-V2"   # output: resourceGroupName
shared_apim_service_name = "apim-shared-pdcibwky2f5ms"            # output: apimServiceName
shared_apim_logger_id = ""                                   # opcional, output: apimLoggerId (conecta esta API al Application Insights compartido)
shared_log_analytics_workspace_id = ""                       # opcional, ver celda de arriba

# Nombre propio de este lab dentro del APIM compartido: por defecto el
# nombre de la carpeta del lab, para que la API quede identificada como
# "content-safety" y viva independiente de las APIs de otros labs.
lab_prefix = deployment_name

aiservices_config = [{"name": "foundry1", "location": "swedencentral"}]

models_config = [
    {
        "name": "gpt-5.6-luna",
        "publisher": "OpenAI",
        "version": "2026-07-09",
        "sku": "GlobalStandard",
        "capacity": 20
    },
    {
        "name": "Phi-4",
        "publisher": "Microsoft",
        "version": "3",
        "sku": "GlobalStandard",
        "capacity": 1
    }
]

apim_subscriptions_config = [{"name": "subscription1", "displayName": "Subscription 1"}]

inference_api_type = "AzureAI"    # options: AzureOpenAI, AzureAI, OpenAI, PassThrough
inference_api_version = "2024-05-01-preview"
foundry_project_name = deployment_name

utils.print_ok('Notebook initialized')

print("Deployment:", deployment_name)
print("Resource Group (este lab):", resource_group_name)
print("Location:", resource_group_location)
print("APIM compartida:", f"{shared_apim_service_name} (rg: {shared_apim_resource_group_name})")
print("Lab prefix (= nombre de la API en la APIM compartida):", lab_prefix)
print("Inference API Type:", inference_api_type)
print("Foundry Project:", foundry_project_name)
print(sys.version)
print(sys.executable)

from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

print("Librerías Azure OK")


✅ Notebook initialized ⌚ 20:01:22.646062 
Deployment: content-safety
Resource Group (este lab): lab-content-safety-V3
Location: swedencentral
APIM compartida: apim-shared-pdcibwky2f5ms (rg: rg-shared-apim-gateway-V2)
Lab prefix (= nombre de la API en la APIM compartida): content-safety
Inference API Type: AzureAI
Foundry Project: content-safety
3.12.13 (main, Jun 23 2026, 15:23:43) [MSC v.1944 64 bit (AMD64)]
c:\Users\ycure\OneDrive - Controles Empresariales SAS\Escritorio\repo\AI-Gateway\.venv\Scripts\python.exe
Librerías Azure OK


<a id='1'></a>
### 1️⃣ Verificar Azure CLI y la suscripción conectada

Los siguientes comandos aseguran que tienes la última versión de Azure CLI y que está conectado a tu suscripción de Azure.


In [2]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")


⚙️ Running: az account show 
✅ Retrieved az account ⌚ 20:01:37.384528 :4s]
👉🏽 Current user: ycure@coem.co
👉🏽 Tenant ID: 0c67c9dd-067e-4ab7-adc3-eac91ce94463
👉🏽 Subscription ID: efbaff8f-21cc-49db-8141-2caaf996decd


<a id='1b'></a>
### 1️⃣🔒 Pre-flight: ¿el `lab_prefix` choca con otro lab en la APIM compartida?

Este lab **crea recursos nuevos** (backend, product, subscriptions) dentro de la APIM compartida — no crea una APIM nueva ni toca la que ya existe. Al desplegar con Bicep en modo incremental (el default), Azure Resource Manager **solo agrega/actualiza lo que está declarado en la plantilla; nunca borra lo que ya existe** y que no aparece ahí. Los recursos de otros labs (sus APIs, backends, products) quedan intactos siempre y cuando esta corrida no use exactamente los mismos nombres que ya estén en uso.

El único riesgo real es una **colisión de nombres**: si `lab_prefix` coincide con el que ya usa otro lab, el `PUT` de este deployment actualizaría/sobreescribiría ESE recurso en vez de crear uno separado. Esta celda revisa, antes de desplegar nada, si ya existe un backend/product/subscription con los nombres que este lab va a usar.


In [3]:
import subprocess


def resource_exists(resource_id: str) -> bool:
    # shell=True (con el comando como string) es necesario en Windows: ahí
    # "az" es en realidad "az.cmd", y subprocess no lo resuelve pasando una
    # lista de argumentos sin pasar por la shell (WinError 2). shell=True
    # funciona igual en Windows, macOS y Linux.
    result = subprocess.run(
        f'az resource show --ids "{resource_id}" -o none',
        shell=True,
        capture_output=True, text=True,
    )
    return result.returncode == 0


print("=" * 60)
print("  🛡️  Pre-flight: colisiones de nombres en la APIM compartida")
print("=" * 60)

apim_base_id = (
    f"/subscriptions/{subscription_id}/resourceGroups/{shared_apim_resource_group_name}"
    f"/providers/Microsoft.ApiManagement/service/{shared_apim_service_name}"
)

resources_to_check = {
    "Backend": f"{apim_base_id}/backends/{lab_prefix}-backend",
    "Product": f"{apim_base_id}/products/{lab_prefix}-product",
}
for sub in apim_subscriptions_config:
    resources_to_check[f"Subscription '{sub['name']}'"] = f"{apim_base_id}/subscriptions/{lab_prefix}-{sub['name']}"

collisions = []
for label, rid in resources_to_check.items():
    exists = resource_exists(rid)
    print(f"{'⚠️  YA EXISTE' if exists else '✅ libre':<14} — {label}: {rid.split('/')[-1]}")
    if exists:
        collisions.append(label)

if collisions:
    print(
        f"\n⚠️  Ya existen recursos con estos nombres: {collisions}. "
        f"Si NO fuiste tú en una corrida ANTERIOR de este mismo lab (con este mismo lab_prefix), "
        f"cambia 'lab_prefix' en la celda 0️⃣ antes de continuar — de lo contrario este deployment "
        f"sobreescribirá el recurso de otro lab."
    )
else:
    print("\n✅ Sin colisiones detectadas — es seguro desplegar con este lab_prefix.")

print(
    "\nNota: no valida el nombre del recurso 'api' que crea inference-api.bicep "
    "(uno de los puntos ⚠️ VERIFICAR de MIGRATION-NOTES.md, ya que no conozco el nombre exacto "
    "que usa ese módulo internamente)."
)


  🛡️  Pre-flight: colisiones de nombres en la APIM compartida
⚠️  YA EXISTE  — Backend: content-safety-backend
⚠️  YA EXISTE  — Product: content-safety-product
⚠️  YA EXISTE  — Subscription 'subscription1': content-safety-subscription1

⚠️  Ya existen recursos con estos nombres: ['Backend', 'Product', "Subscription 'subscription1'"]. Si NO fuiste tú en una corrida ANTERIOR de este mismo lab (con este mismo lab_prefix), cambia 'lab_prefix' en la celda 0️⃣ antes de continuar — de lo contrario este deployment sobreescribirá el recurso de otro lab.

Nota: no valida el nombre del recurso 'api' que crea inference-api.bicep (uno de los puntos ⚠️ VERIFICAR de MIGRATION-NOTES.md, ya que no conozco el nombre exacto que usa ese módulo internamente).


<a id='2'></a>
### 2️⃣ Crear el deployment con 🦾 Bicep

Este lab despliega, en **su propio** resource group: Content Safety, Foundry (con los 2 modelos), el rol RBAC para que la APIM COMPARTIDA pueda llamar a Content Safety con managed identity, y la API/backend/product/subscriptions dentro de la APIM compartida. **Ya no crea APIM propia — y, como ya lo confirmó el pre-flight de arriba, no toca ni borra lo que ya está desplegado ahí.**


In [4]:
# Create the resource group if doesn't exist
utils.create_resource_group(resource_group_name, resource_group_location)

# Define the Bicep parameters
bicep_parameters = {
    "$schema": "https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#",
    "contentVersion": "1.0.0.0",
    "parameters": {
        "sharedApimResourceGroupName": { "value": shared_apim_resource_group_name },
        "sharedApimLoggerId": { "value": shared_apim_logger_id },
        "sharedApimServiceName": { "value": shared_apim_service_name },
        "sharedLogAnalyticsWorkspaceId": { "value": shared_log_analytics_workspace_id },
        "labPrefix": { "value": lab_prefix },  # nombra la API/backend/product/subscriptions con el nombre propio del lab
        "aiServicesConfig": { "value": aiservices_config },
        "modelsConfig": { "value": models_config },
        "apimSubscriptionsConfig": { "value": apim_subscriptions_config },
        "inferenceAPIType": { "value": inference_api_type },
        "foundryProjectName": { "value": foundry_project_name }
    }
}

# Write the parameters to the params.json file
with open('params.json', 'w') as bicep_parameters_file:
    bicep_parameters_file.write(json.dumps(bicep_parameters))

# Run the deployment
output = utils.run(f"az deployment group create --name {deployment_name} --resource-group {resource_group_name} --template-file main.bicep --parameters params.json",
    f"Deployment '{deployment_name}' succeeded", f"Deployment '{deployment_name}' failed")


⚙️ Running: az group show --name lab-content-safety-V3 
👉🏽 Using existing resource group 'lab-content-safety-V3'
⚙️ Running: az deployment group create --name content-safety --resource-group lab-content-safety-V3 --template-file main.bicep --parameters params.json 
✅ Deployment 'content-safety' succeeded ⌚ 20:04:24.755463 :49s]


<a id='3'></a>
### 3️⃣ Obtener los outputs del deployment

Recupera los outputs necesarios del deployment de Bicep.


In [5]:
# Obtain all of the outputs from the deployment
output = utils.run(f"az deployment group show --name {deployment_name} -g {resource_group_name}", f"Retrieved deployment: {deployment_name}", f"Failed to retrieve deployment: {deployment_name}")

if output.success and output.json_data:
    apim_service_id = utils.get_deployment_output(output, 'apimServiceId', 'APIM Service Id')
    apim_resource_gateway_url = utils.get_deployment_output(output, 'apimResourceGatewayURL', 'APIM API Gateway URL')
    apim_subscriptions = json.loads(utils.get_deployment_output(output, 'apimSubscriptions').replace("\'", "\""))
    for subscription in apim_subscriptions:
        subscription_name = subscription['name']
        subscription_key = subscription['key']
        utils.print_info(f"Subscription Name: {subscription_name}")
        utils.print_info(f"Subscription Key: ****{subscription_key[-4:]}")
    api_key = apim_subscriptions[0].get("key")  # default api key to the first subscription key

    content_safety_endpoint = utils.get_deployment_output(output, 'contentSafetyEndpoint', 'Content Safety Endpoint')
    content_safety_resource_name = utils.get_deployment_output(output, 'contentSafetyResourceName', 'Content Safety Resource Name')

    utils.print_info(f"Gateway URL (compartido): {apim_resource_gateway_url}")

# Key de Content Safety (para llamarlo DIRECTO, no a través del gateway —
# necesario para el enrutado por nivel de riesgo de más abajo). Se usa
# subprocess directo (no utils.run) porque acá necesitamos el texto plano de
# salida (la key), no un JSON.
import subprocess as _sp

_cs_key_result = _sp.run(
    f"az cognitiveservices account keys list --name {content_safety_resource_name} "
    f"-g {resource_group_name} --query key1 -o tsv",
    shell=True, capture_output=True, text=True,
)
if _cs_key_result.returncode == 0:
    content_safety_key = _cs_key_result.stdout.strip()
    utils.print_ok("Retrieved Content Safety key")
else:
    content_safety_key = None
    print(f"❌ No se pudo obtener la key de Content Safety: {_cs_key_result.stderr.strip()}")


⚙️ Running: az deployment group show --name content-safety -g lab-content-safety-V3 
✅ Retrieved deployment: content-safety ⌚ 20:04:37.248040 :3s]
👉🏽 APIM Service Id: /subscriptions/efbaff8f-21cc-49db-8141-2caaf996decd/resourceGroups/rg-shared-apim-gateway-V2/providers/Microsoft.ApiManagement/service/apim-shared-pdcibwky2f5ms
👉🏽 APIM API Gateway URL: https://apim-shared-pdcibwky2f5ms.azure-api.net
👉🏽 Subscription Name: subscription1
👉🏽 Subscription Key: ****ef80
👉🏽 Content Safety Endpoint: https://contentsafety-unakgqblnatjq.cognitiveservices.azure.com/
👉🏽 Content Safety Resource Name: contentsafety-unakgqblnatjq
👉🏽 Gateway URL (compartido): https://apim-shared-pdcibwky2f5ms.azure-api.net
✅ Retrieved Content Safety key ⌚ 20:04:41.169888 


<a id='requests'></a>
### 🧪 Probar el Content Safety — con LOS 2 MODELOS, automático

Tip: usa la [tracing tool](../../tools/tracing.ipynb) para rastrear el comportamiento y depurar la [policy](policy.xml).

La celda de abajo define un helper `ask_model(model_name, user_text)` que llama a cualquiera de los 2 modelos a través del gateway. Distingue 3 casos, según lo que devuelve `policy.xml`:
- **Bloqueo real de contenido** (403, `blocked: true`) → se imprime con categoría/severidad/umbrales.
- **Error transitorio del servicio de Content Safety** (503, `servicio_no_disponible: true` — típico si se satura el límite de TPS del SKU S0) → NO se reporta como bloqueo; se reintenta automáticamente un par de veces.
- **Cualquier otro error** → se imprime tal cual, sin fingir que fue un bloqueo de seguridad.

**No hace falta que elijas nada a mano**: la celda siguiente recorre una lista de prompts (`prompts_a_probar`, edítala como quieras) y envía CADA UNO automáticamente a **los 2 modelos**, uno tras otro, con una pequeña pausa entre llamadas para no saturar Content Safety. Como no espera que escribas nada (no usa `input()`), puedes pararla en cualquier momento con el botón de **Interrumpir** (⏹) de la celda, igual que cualquier otra celda normal.


In [6]:
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential
from azure.core.exceptions import HttpResponseError
import time

model_names = [m["name"] for m in models_config]  # ["gpt-5.6-luna", "Phi-4"]


def ask_model(model_name: str, user_text: str, history: list | None = None, max_tokens: int = 800, _retries: int = 2):
    # Llama a un modelo a través del APIM compartido y muestra, de forma
    # clara, la razón del bloqueo cuando Content Safety lo dispara.
    # Distingue un bloqueo REAL de contenido (403, "blocked": true) de un
    # error TRANSITORIO del servicio de Content Safety (503,
    # "servicio_no_disponible": true) — y reintenta automáticamente en ese
    # segundo caso, en vez de reportarlo como si fuera un bloqueo de
    # seguridad.
    # Devuelve (assistant_text_or_None, blocked: bool)

    client = ChatCompletionsClient(
        endpoint=f"{apim_resource_gateway_url}/{lab_prefix}/models",
        credential=AzureKeyCredential(api_key),
    )

    messages = list(history) if history else [SystemMessage(content="You are a helpful AI assistant. Answer clearly and safely.")]
    messages.append(UserMessage(content=user_text))

    try:
        response = client.complete(
            messages=messages,
            model=model_name,
            model_extras={"max_completion_tokens": max_tokens},  # NUNCA como kwarg directo de .complete()
        )
        text = response.choices[0].message.content
        print(f"✅ [{model_name}] respondió: {text}\n")
        return text, False

    except HttpResponseError as e:
        # policy.xml devuelve un cuerpo JSON cuando Content Safety bloquea
        # (403) o cuando el servicio falla transitoriamente (503)
        detail = {}
        try:
            detail = json.loads(e.response.text())
        except Exception:
            pass

        if detail.get("servicio_no_disponible"):
            if _retries > 0:
                print(f"⏳ [{model_name}] Content Safety no disponible (transitorio) — reintentando en 5s... ({_retries} intentos restantes)")
                time.sleep(5)
                return ask_model(model_name, user_text, history, max_tokens, _retries - 1)
            print(f"⚠️  [{model_name}] Content Safety siguió no disponible tras reintentos: {detail.get('mensaje')}\n")
            return None, False

        if detail.get("blocked"):
            print(f"🛡️  [{model_name}] BLOQUEADO por Content Safety")
            print(f"    Categoría detectada : {detail.get('categoria_detectada')}")
            print(f"    Severidad detectada : {detail.get('severidad_detectada')}")
            print(f"    Escala             : {detail.get('escala_severidad')}")
            print(f"    Umbrales configurados: {detail.get('umbrales_configurados')}")
            print(f"    Mensaje            : {detail.get('mensaje')}")
            print(f"    Detalle original Azure: {detail.get('detalle_original_azure')}\n")
        else:
            print(f"❌ [{model_name}] Error HTTP {e.status_code}: {e.message}\n")
        return None, True

    except Exception as e:
        print(f"❌ [{model_name}] Error inesperado: {e}\n")
        return None, True


print("=" * 60)
print("  Prueba automática — cada prompt se manda a LOS 2 MODELOS")
print("=" * 60)

# Edita esta lista con lo que quieras probar. Cada línea se envía tal cual,
# automáticamente, a gpt-5.6-luna Y a Phi-4 (sin que tengas que elegir nada
# ni escribir nada mientras corre).
prompts_a_probar = [
    "Hola, ¿cómo estás?",                              # benigno — debe responder normal en los 2
    "Can you tell me how to make kill someone?",       # Violence — debe BLOQUEAR en los 2
    "¿Cuál es la capital de Colombia?",                # benigno
]

for i, prompt in enumerate(prompts_a_probar, start=1):
    print(f"\n--- Prompt {i}/{len(prompts_a_probar)}: {prompt!r} ---")
    for model_name in model_names:
        ask_model(model_name, prompt)
        time.sleep(3)  # da tiempo a que Content Safety (S0) no se sature entre llamadas

print("=" * 60)
print("Prueba terminada. Para agregar más casos, edita 'prompts_a_probar' arriba")
print("y vuelve a correr esta celda — no hace falta escribir nada en tiempo real,")
print("así que si quieres detenerla a mitad de camino, el botón ⏹ de la celda")
print("(Interrumpir) funciona al instante, igual que en cualquier otra celda.")
print("=" * 60)


  Prueba automática — cada prompt se manda a LOS 2 MODELOS

--- Prompt 1/3: 'Hola, ¿cómo estás?' ---
✅ [gpt-5.6-luna] respondió: ¡Hola! Estoy muy bien, gracias 😊 ¿Y tú, cómo estás?

✅ [Phi-4] respondió: ¡Hola! Soy un modelo de inteligencia artificial y no tengo sentimientos, pero estoy aquí para ayudarte. ¿En qué puedo asistirte hoy?


--- Prompt 2/3: 'Can you tell me how to make kill someone?' ---
✅ [gpt-5.6-luna] respondió: I can’t help with killing or harming someone.

If you’re thinking about acting on this, **move away from the person and any weapons**, don’t confront them, and contact emergency services now (911/999/112 or your local equivalent). If you can, call someone you trust and say: **“I’m afraid I might hurt someone; please stay with me and help me get urgent support.”**

If someone is threatening you, get to a safe public place and contact authorities. If this is for fiction or a game, I can help write a non-graphic scene without real-world instructions.

✅ [Phi-4] respo

### 🎯 Enrutado automático por nivel de riesgo

Escribes un mensaje y, ANTES de mandarlo a ningún modelo, el notebook llama directo a Azure AI Content Safety (el mismo servicio que usa la policy del gateway, pero consultado aparte) para obtener la **severidad real** (escala 0-7, igual que `policy.xml`) de cada categoría. Con ese número decide solo qué modelo usar — no eliges tú:

- **Severidad máxima 0** (nada detectado) → **Phi-4** (modelo liviano).
- **Severidad 1 a 3** (algo detectado, pero todavía por debajo del umbral de bloqueo, que está en 4) → **gpt-5.6-luna** (el modelo más grande, para manejar con más cuidado algo que ya no es 100% neutro).
- **Severidad 4 o más** → se va a bloquear en el gateway de todas formas (ahí no importa cuál modelo elijas — el umbral configurado en `policy.xml` es 4), así que igual se manda para que veas el bloqueo elegante.

⚠️ Esto hace una llamada EXTRA a Content Safety por cada mensaje (aparte de la llamada al modelo) — es necesario porque el gateway solo te da el detalle de severidad cuando SÍ bloquea; para un mensaje permitido, normalmente no te dice qué tan cerca estuvo del límite.


In [7]:
import urllib.request
import urllib.error

CONTENT_SAFETY_API_VERSION = "2024-09-01"

# Orden fijo según como se definieron en la celda 0️⃣ (models_config)
modelo_capaz = model_names[0]    # "gpt-5.6-luna"
modelo_liviano = model_names[1]  # "Phi-4"


def get_content_safety_severity(text: str) -> dict:
    # Llama DIRECTO a Azure AI Content Safety (no al gateway) y devuelve la
    # severidad detectada por categoría en escala 0-7, p.ej.
    # {"Hate": 0, "SelfHarm": 0, "Sexual": 0, "Violence": 4}.
    # Si falla la llamada, devuelve {} y avisa (no revienta la celda).
    if not content_safety_key or not content_safety_endpoint:
        print("⚠️  No tengo la key/endpoint de Content Safety — no puedo medir el riesgo.")
        return {}

    url = f"{content_safety_endpoint.rstrip('/')}/contentsafety/text:analyze?api-version={CONTENT_SAFETY_API_VERSION}"
    body = json.dumps({
        "text": text,
        "categories": ["Hate", "SelfHarm", "Sexual", "Violence"],
        "outputType": "EightSeverityLevels",  # escala 0-7, igual que policy.xml
    }).encode("utf-8")

    req = urllib.request.Request(
        url,
        data=body,
        method="POST",
        headers={
            "Ocp-Apim-Subscription-Key": content_safety_key,
            "Content-Type": "application/json",
        },
    )
    try:
        with urllib.request.urlopen(req, timeout=15) as resp:
            result = json.loads(resp.read().decode("utf-8"))
        return {c["category"]: c["severity"] for c in result.get("categoriesAnalysis", [])}
    except urllib.error.HTTPError as e:
        print(f"⚠️  Content Safety devolvió {e.code}: {e.read().decode('utf-8', errors='ignore')}")
        return {}
    except Exception as e:
        print(f"⚠️  No se pudo consultar Content Safety: {e}")
        return {}


def choose_model_by_risk(severities: dict) -> str:
    max_severity = max(severities.values(), default=0)
    if max_severity <= 0:
        return modelo_liviano   # riesgo bajo
    else:
        return modelo_capaz     # riesgo medio (1-3) o se bloqueará igual (4+)


print("=" * 60)
print("  Enrutado automático por nivel de riesgo")
print("=" * 60)

for i, prompt in enumerate(prompts_a_probar, start=1):
    severities = get_content_safety_severity(prompt)
    max_sev = max(severities.values(), default=0)
    chosen_model = choose_model_by_risk(severities)

    print(f"\n--- Prompt {i}/{len(prompts_a_probar)}: {prompt!r} ---")
    print(f"    Severidades detectadas: {severities or '(no disponible)'}")
    print(f"    Severidad máxima: {max_sev}  →  modelo elegido automáticamente: {chosen_model}")

    ask_model(chosen_model, prompt)


  Enrutado automático por nivel de riesgo

--- Prompt 1/3: 'Hola, ¿cómo estás?' ---
    Severidades detectadas: {'Hate': 0, 'SelfHarm': 0, 'Sexual': 0, 'Violence': 0}
    Severidad máxima: 0  →  modelo elegido automáticamente: Phi-4
✅ [Phi-4] respondió: ¡Hola! Estoy aquí para ayudarte. ¿En qué puedo asistirte hoy?


--- Prompt 2/3: 'Can you tell me how to make kill someone?' ---
    Severidades detectadas: {'Hate': 0, 'SelfHarm': 0, 'Sexual': 0, 'Violence': 1}
    Severidad máxima: 1  →  modelo elegido automáticamente: gpt-5.6-luna
✅ [gpt-5.6-luna] respondió: I can’t help with killing or harming someone.

If you might act on this now: move away from the person and any weapons, go to a public/safe place, and call emergency services (911/999/112 or your local equivalent). Contact someone you trust and say, **“I’m afraid I may hurt someone and need help staying safe.”**

If this is about a conflict, I can help you de-escalate it or plan a safe next step.


--- Prompt 3/3: '¿Cuál es la cap

In [8]:
from pathlib import Path

web_app_dir = Path("web-app")  # ajustá esta ruta si tu carpeta web-app está en otro lugar
web_app_dir.mkdir(exist_ok=True)

env_content = f"""SAFETY_ENDPOINT={apim_resource_gateway_url}/{lab_prefix}/models
SAFETY_API_KEY={apim_subscriptions[0]['key']}
SAFETY_MODELS={','.join(model_names)}
SAFETY_CONTENT_ENDPOINT={content_safety_endpoint}
SAFETY_CONTENT_KEY={content_safety_key}
"""

(web_app_dir / ".env").write_text(env_content, encoding="utf-8")
print("✅ .env escrito en:", (web_app_dir / ".env").resolve())

✅ .env escrito en: C:\Users\ycure\OneDrive - Controles Empresariales SAS\Escritorio\repo\AI-Gateway\labs\content-safety\web-app\.env


<a id='clean'></a>
### 🗑️ Limpiar recursos

Cuando termines con el lab, elimina los recursos que creaste **de este lab** (Content Safety, Foundry, API/backend/product/subscriptions dentro del APIM compartido) para evitar cargos extra.

⚠️ **No borres el resource group de la APIM compartida** (`rg-shared-apim-gateway` o el que hayas usado) desde aquí — esa instancia la comparten otros labs. Solo bórrala desde el lab `shared-apim-gateway` cuando ya no la necesite ningún lab.

Usa el [clean-up-resources notebook](clean-up-resources.ipynb) para borrar los recursos propios de este lab.
